# Script Review — Kuku submission edition · 2.1 + real-output export
**Problem 3 · Qwen3-14B · Kaggle T4 ×2 · Gradio reviewer**

This preserves your uploaded application. A new section after the engineering tests generates test ZIPs and runs real Qwen inference on a small original six-case set, then exports raw model calls, review tables and a standalone HTML report. The Gradio app still launches near the end, with an optional no-login PUBLIC_DEMO mode. Do not claim new GPU accuracy before inspecting the downloaded results.

**Quick run:** Enable Kaggle Internet + GPU T4×2. Run all cells. The new submission stage executes six cases; the Gradio app then starts and its URL appears in the app-log cell. A live Gradio link is temporary and requires a running Kaggle session. Save a *public, cleared* Kaggle notebook version as your durable prototype link; never publish confidential scripts or passwords.

**Longer runs:** Set `SUBMISSION_PACK='hard10'` (ten difficult cases) or `'full18'` (eighteen cases). These are independent short fictional scripts, not ten consecutive episodes of one show. To run your own scripts in Gradio, clear demo text, upload a ZIP, set episode IDs, and Analyze.

# Script Review — submission notebook · 2.1 · presentation edition
**Problem 3 · Qwen3-14B · Kaggle T4 ×2 · Gradio editorial review app**

This notebook contains the entire source package. No previous notebook or patch is required. It extracts concrete decision, identity and danger setups, verifies questions against the full episode, and lets an editor approve, edit, reject or add hooks. It does not train a model or track later-episode payoffs.

## Start here
1. Select **GPU T4 ×2** and enable **Internet** in Kaggle.
2. The default now **skips batch inference and launches the Gradio app**. Upload your ten-episode ZIP directly in the app, clear pasted text, enter `3,9` under **Uploaded episode IDs**, and keep Maximum episodes at 2. No Add Input step is required for app uploads.
3. Run cells in order; run the app-log cell again if the link is not visible yet. Analyze the two selected episodes, then download the run ZIP from Review & export. Model quality has not been validated for this new revision.
4. To use file upload and editorial review in a browser, launch the optional Gradio app near the end. Do not run a batch while that app holds GPU memory.

The app, original examples, earlier real synthetic-control outputs, a two-page solution document and interview walkthrough are included. Do not publish Kuku scripts or archives containing them. Use the original demos for public outputs and clear notebook outputs before making a public copy.

## 1. Configuration
The model is unchanged. The new pipeline is `desk-2.1`. Defaults keep the first run small. `episode_ids=[]` selects the first `max_episodes` episodes from a dataset. Your ten-episode pack parses as timed scripts; episodes 3 and 9 each need two verification chunks. Plain-text scripts use the final 12 non-empty lines; timed scripts use the final 45 seconds.
**Version 2.1 changes:** shorter constrained verification, clearer identity-mystery instructions, live elapsed status, readable evidence cards, episode-ID selection, and Excel-friendly CSVs. Fresh GPU speed and quality remain unmeasured.

**Presentation edition:** Product renamed Script Review; refreshed dark UI; ten original challenge demos with a separate semantic rubric. Inference is unchanged from 2.1. Clear uploaded files and episode IDs before trying a pasted demo.


In [ ]:
from pathlib import Path
import os,sys,json,subprocess,venv,urllib.request
# Release this notebook's previous app before overwriting its source.
if 'APP_PROCESS' in globals() and APP_PROCESS.poll() is None:
    APP_PROCESS.terminate()
    try: APP_PROCESS.wait(timeout=20)
    except subprocess.TimeoutExpired:
        APP_PROCESS.kill(); APP_PROCESS.wait()
ROOT=Path('/kaggle/working/cliffhanger_desk')
SRC=ROOT/'src'
ROOT.mkdir(parents=True,exist_ok=True)
CONFIG={
    'model_id':'Qwen/Qwen3-14B',
    'revision':'40c069824f4251a91eefaf281ebe4c544efd3e18',
    'input_mode':'dataset',  # Change to 'dataset' for attached scripts.
    'input_dir':'/kaggle/input',  # Prefer your specific dataset subfolder.
    'output_root':str(ROOT/'runs'),
    'episode_ids':[3,9],
    'max_episodes':2,
    'ending_seconds':45,
    'plain_ending_blocks':12,
    'verification_chunk_chars':9000,
    'max_verification_chunks':6,
    'thinking':True,
    'verification_thinking':False,  # Test shorter constrained verification.

    'max_input_tokens':8192,
    'thinking_tokens':2048,
    'json_tokens':650,
    'seed':42,
}
RUN_BATCH=False
RUN_EVALUATION=False  # Eight original development fixtures; extra GPU time.
print(json.dumps(CONFIG,indent=2))


## 2. Unpack the included source
This cell writes source files, original examples and documentation. It contains no Kuku script content. The source ZIP is also available for downloading in the final cell.

In [ ]:
import base64,io,zipfile
SOURCE_ARCHIVE='...'
SRC.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ARCHIVE))) as z:
    for member in z.infolist():
        destination=(SRC/member.filename).resolve()
        if not destination.is_relative_to(SRC.resolve()):
            raise ValueError('Invalid bundled path')
        if not member.is_dir():
            destination.parent.mkdir(parents=True,exist_ok=True)
            destination.write_bytes(z.read(member))
print('Source ready:',SRC)


## 3. Install the isolated environment
Targets ordinary CPython 3.10-3.13. Uses the manual pip bootstrap that worked in your earlier environment and installs `wrapt` early. A startup-hook warning can appear before that install completes.

All packages live inside this project's environment. `huggingface-hub==0.36.2` is deliberately compatible with both pinned Transformers and Gradio. The model revision remains unchanged. Initial dependency and weight downloads require several GB and available disk space.

In [ ]:
if not (3,10)<=sys.version_info[:2]<=(3,13):
    raise RuntimeError('Use Python 3.10–3.13 for this pinned setup.')
ENV=ROOT/'env'
venv.EnvBuilder(with_pip=False,system_site_packages=False).create(ENV)
PYTHON=str(ENV/'bin/python')
CHILD_ENV=os.environ.copy()
for key in ('PYTHONPATH','PYTHONHOME'):CHILD_ENV.pop(key,None)
CHILD_ENV.update(PYTHONNOUSERSITE='1',TOKENIZERS_PARALLELISM='false',GRADIO_ANALYTICS_ENABLED='False')
def run_worker(args):
    p=subprocess.Popen([PYTHON,'-u',*args],env=CHILD_ENV,cwd=str(SRC),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout:print(line,end='',flush=True)
        if p.wait():raise RuntimeError('Subprocess failed. Read the error above before continuing.')
    except BaseException:
        if p.poll() is None:p.terminate();p.wait()
        raise
if subprocess.run([PYTHON,'-m','pip','--version'],env=CHILD_ENV,capture_output=True).returncode:
    bootstrap=ROOT/'get-pip.py'
    urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py',bootstrap)
    run_worker([str(bootstrap)])
run_worker(['-m','pip','install','--disable-pip-version-check','wrapt==1.17.2'])
run_worker(['-m','pip','install','--disable-pip-version-check','-r',str(SRC/'requirements.txt')])
run_worker(['-m','pip','check'])
run_worker(['-c',"import torch,gradio,wrapt,bitsandbytes; from transformers import Qwen3ForCausalLM; from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn; print('Imports OK. Gradio:',gradio.__version__); print('GPUs:',torch.cuda.device_count()); assert torch.cuda.device_count()>=2, 'Select GPU T4 x2 before inference'"])


## 4. Local engineering checks — no model download
These use stubbed responses to exercise parsing, evidence IDs, review/export behavior and app construction. Passing them is not an accuracy claim.

In [ ]:
run_worker([str(SRC/'tests.py')])


## Submission run: original test cases + real Qwen outputs

This is the additional submission workflow. It is **actual Qwen3-14B inference** when `RUN_SUBMISSION_TESTS=True`; it is **not** a replay of hard-coded predictions. It runs on six original fictional cases by default and includes intentionally closed cases and traps. Change `SUBMISSION_PACK` to `hard10` or `full18` after examining the six-case result.

**Privacy:** Only original test scripts are used here. The full exported run contains raw prompts, source text, output text, timestamps and the model revision. Do not put confidential Kuku scripts in a public run or repo.

Run the configuration/unpack/install/test cells first. After this block, you can continue to the Gradio app launch cell to show the interactive experience. These batch runs and the app run sequentially so their GPU processes never overlap.


In [ ]:
from pathlib import Path
import zipfile, csv, sys
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
from evaluation import CASES
from hard_cases import HARD_CASES
ALL_TEST_CASES={c['id']:c for c in CASES+HARD_CASES}
PACK_SELECTIONS={'quick6':[101,102,201,205,206,209],
                 'hard10':[c['id'] for c in HARD_CASES],
                 'full18':[c['id'] for c in CASES+HARD_CASES]}
PACK_DIR=ROOT/'submission_input_packs'
PACK_DIR.mkdir(parents=True,exist_ok=True)
for name,selected in PACK_SELECTIONS.items():
    target=PACK_DIR/(name+'.zip')
    with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
        for cid in selected:
            z.writestr(f'episode_{cid}.txt',ALL_TEST_CASES[cid]['text'].rstrip()+'\n')
        z.writestr('README.md','Original synthetic fictional scripts, not model output. Each file is a separate case.\n')
    print(name,len(selected),'scripts:',target)
RUBRIC_PATH=ROOT/'SUBMISSION_EXPECTED_RUBRIC.csv'
with RUBRIC_PATH.open('w',newline='',encoding='utf-8-sig') as f:
    writer=csv.writer(f)
    writer.writerow(['episode_id','title','expected_open_kinds','semantic_check'])
    for case in ALL_TEST_CASES.values():
        writer.writerow([case['id'],case['title'],';'.join(case['expected']),case.get('rubric','Check exact question and cited evidence')])
print('Expected labels for HUMAN CHECK ONLY (not given to Qwen):', RUBRIC_PATH)


In [ ]:
# Three options: 'quick6' (recommended), 'hard10', 'full18'.
SUBMISSION_PACK='quick6'
RUN_SUBMISSION_TESTS=True
if RUN_SUBMISSION_TESTS:
    if 'APP_PROCESS' in globals() and APP_PROCESS.poll() is None:
        raise RuntimeError('Stop the app first; do not run two GPU model processes.')
    submission_cfg=dict(CONFIG)
    submission_cfg.update(input_mode='dataset', input_dir=str(PACK_DIR/SUBMISSION_PACK),
                          episode_ids=[], max_episodes=len(PACK_SELECTIONS[SUBMISSION_PACK]))
    # Give the batch exactly one uploaded ZIP in its own folder.
    SUBMISSION_INPUT_DIR=ROOT/'submission_input_active'
    SUBMISSION_INPUT_DIR.mkdir(parents=True,exist_ok=True)
    for old in SUBMISSION_INPUT_DIR.iterdir():
        if old.is_file():old.unlink()
    import shutil
    shutil.copy2(PACK_DIR/(SUBMISSION_PACK+'.zip'),SUBMISSION_INPUT_DIR/(SUBMISSION_PACK+'.zip'))
    submission_cfg['input_dir']=str(SUBMISSION_INPUT_DIR)
    SUBMISSION_CONFIG_PATH=ROOT/'config.submission.json'
    SUBMISSION_CONFIG_PATH.write_text(json.dumps(submission_cfg,indent=2))
    print('Running REAL Qwen inference on',SUBMISSION_PACK,':',PACK_SELECTIONS[SUBMISSION_PACK])
    run_worker([str(SRC/'run_batch.py'),'--config',str(SUBMISSION_CONFIG_PATH)])
    SUBMISSION_RUN=Path((Path(CONFIG['output_root'])/'latest_run.txt').read_text().strip())
    print('Exported output:',SUBMISSION_RUN)
    from IPython.display import display,FileLink
    display(FileLink(str(SUBMISSION_RUN)+'.zip'))
    display(FileLink(str(SUBMISSION_RUN/'review.html')))
    display(FileLink(str(SUBMISSION_RUN/'review.csv')))
    display(FileLink(str(SUBMISSION_RUN/'episode_summary.csv')))
    # Compare coarse categories; not an independent semantic accuracy score.
    output=json.loads((SUBMISSION_RUN/'results.json').read_text())
    print('\nCATEGORY-ONLY SANITY CHECK (not accuracy claim):')
    match=0
    for rec in output['episodes']:
        case=ALL_TEST_CASES[rec['episode_id']]
        expected=set(case['expected'])
        got={c['kind'] for c in rec['candidates'] if c['status']=='open_candidate'}
        good=expected==got and not rec['errors']
        match+=bool(good)
        print(rec['episode_id'],case['title'],'expected:',sorted(expected),'observed:',sorted(got),
              'MATCH' if good else 'REVIEW', 'errors:',len(rec['errors']))
    print('CATEGORY MATCH',match,'/',len(output['episodes']))
    print('REVIEW THE QUESTIONS AND EVIDENCE: category agreement is not semantic validation.')
else:
    print('Skipped. Set RUN_SUBMISSION_TESTS=True to produce actual model outputs.')


## 5. Batch inference
Uploads are optional for demo mode. For dataset mode, missing inputs are detected before model loading. This is actual Qwen3 inference; no example answers are substituted.

Extraction uses thinking and returns typed story slots. Verification now uses constrained non-thinking JSON by default; set verification_thinking=True to compare the old reasoning mode. Code constructs questions. Verification covers the entire episode in chunks. Invalid formatting, overlong strings or repetition get at most one constrained retry. An unresolved proposal still requires editorial review. An error or an empty result must not be interpreted as a closed ending.

Each run has its own output folder. No old generated answers are loaded as new predictions. Model weights may be reused from the download cache.

In [ ]:
if 'APP_PROCESS' in globals() and APP_PROCESS.poll() is None:
    raise RuntimeError('Stop the Gradio app with the stop cell before running a batch.')
CONFIG_PATH=ROOT/'config.runtime.json'
CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2))
if RUN_BATCH:
    run_worker([str(SRC/'run_batch.py'),'--config',str(CONFIG_PATH)])
else:
    print('Batch skipped. You can launch the app below instead.')


## 6. Inspect and download the latest run
The HTML report works without a GPU. The ZIP includes scripts, raw calls and human reviews; keep confidential runs private. `baseline_controls_v11` is historical evidence, not the output of this run.

In [ ]:
from IPython.display import display,FileLink
latest=Path(CONFIG['output_root'])/'latest_run.txt'
if latest.exists():
    LAST_RUN=Path(latest.read_text().strip())
    print('Latest run:',LAST_RUN.name)
    for filename in ('run_metrics.json','evaluation.json'):
        file=LAST_RUN/filename
        if file.exists():print(file.read_text())
    for file in (LAST_RUN/'review.html',LAST_RUN/'review.csv',LAST_RUN/'episode_summary.csv',LAST_RUN/'results.json',Path(str(LAST_RUN)+'.zip')):
        if file.exists():display(FileLink(str(file)))
else:
    print('No batch run saved yet. Use the app or run the batch cell.')


## 7. Optional regression evaluation
Eight original development cases test the presence or absence of open hook categories. They are visible in the source and are not a held-out benchmark. The output explicitly labels the limited metric. This cell loads the model in a new process; it does not reuse the previous batch process.

In [ ]:
if RUN_EVALUATION:
    if 'APP_PROCESS' in globals() and APP_PROCESS.poll() is None:
        raise RuntimeError('Stop the Gradio app before running evaluation.')
    CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2))
    run_worker([str(SRC/'run_batch.py'),'--config',str(CONFIG_PATH),'--evaluate'])
else:
    print('Evaluation skipped. Set RUN_EVALUATION=True in configuration to run it.')


## 8. Optional Gradio review app
`LAUNCH_APP=True` is enabled below. Run this cell. The app starts in the background; the next cell shows its log and temporary URL. Model loading is lazy and occurs on the first Analyze request. Later requests reuse the model.

The app has a file-upload control, pasted text, original demos, evidence review, editable decisions, manual missed-hook entry, downloads and evaluation. Share links require Internet and a running Kaggle session. They are temporary, not permanent hosting; submit the notebook/repo as the durable prototype link. If Kaggle blocks the tunnel, batch inference and exports still work.

`PUBLIC_DEMO=True` now makes a publicly accessible no-login link on a separate **public_demo** output directory and hides the private Kaggle dataset button. Use only original fictional/public data in this mode. To use private scripts, set `PUBLIC_DEMO=False` to enable the generated password and avoid publicly sharing those credentials.

In [ ]:
LAUNCH_APP=True
PUBLIC_DEMO=True  # Recommended for judges; use only original public scripts.
if LAUNCH_APP:
    if 'APP_PROCESS' in globals() and APP_PROCESS.poll() is None:
        print('App is already running. Use the next cell for its log.')
    else:
        import secrets
        CONFIG_PATH=ROOT/'config.runtime.json'
        CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2))
        APP_PASSWORD='' if PUBLIC_DEMO else secrets.token_urlsafe(18)
        app_env=dict(CHILD_ENV,DESK_PASSWORD=APP_PASSWORD)
        APP_LOG=ROOT/'app.log'
        with APP_LOG.open('w') as log:
            APP_PROCESS=subprocess.Popen([PYTHON,'-u',str(SRC/'app.py'),'--config',str(CONFIG_PATH),'--share']+(['--public-demo'] if PUBLIC_DEMO else []),cwd=str(SRC),env=app_env,stdout=log,stderr=subprocess.STDOUT,text=True)
        print('App starting. Run the next cell to see the URL.')
        if PUBLIC_DEMO:
            print('Public demo: NO password; use only public/original scripts.')
        else:
            print('Username: reviewer')
            print('Password:',APP_PASSWORD)
else:
    print('App launch skipped. Set LAUNCH_APP=True above when ready.')


In [ ]:
app_log=ROOT/'app.log'
if app_log.exists():print(app_log.read_text()[-12000:])
else:print('Launch the app first.')
if 'APP_PROCESS' in globals():print('Process running:',APP_PROCESS.poll() is None)


## 9. Stop the app when finished
This cell is disabled by default so Run All does not immediately shut the app down. Set STOP_APP=True when you want to release its GPU memory.

In [ ]:
STOP_APP=False
if STOP_APP and 'APP_PROCESS' in globals() and APP_PROCESS.poll() is None:
    APP_PROCESS.terminate()
    try:APP_PROCESS.wait(timeout=20)
    except subprocess.TimeoutExpired:APP_PROCESS.kill();APP_PROCESS.wait()
    print('App stopped.')


## 10. Download source and earlier submission document
Read CHANGES_2_1.md for this update. The included PDF describes the earlier submission and is not validation of 2.1. The source package and original examples are safe to publish. The code below packages only the included source; it does not include your runtime configuration, password, private inputs or run folders.

Before submitting, generate real outputs with this version and inspect them. Include a clear account of failures in your solution document and walkthrough. The authoring tests do not establish new GPU accuracy.

In [ ]:
import shutil
source_archive=shutil.make_archive(str(ROOT/'Cliffhanger_Desk_Source'),'zip',str(SRC))
display(FileLink(source_archive))
display(FileLink(str(SRC/'Cliffhanger_Desk_Solution.pdf')))
print('README and walkthrough:',SRC)
